<a href="https://colab.research.google.com/github/hussainturii/Advanced-Computer-Vision---Deep-Learning/blob/main/ViTs%20from%20scratch%20using%20Pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [60]:
#import libraries
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn

In [61]:
#transformation od PI: data into tensor format
transformation_operation = transforms.Compose([transforms.ToTensor()])

In [62]:
#variables
batch_size = 64
img_size = 28
patch_size = 7
num_channels = 1
num_patches = (img_size // patch_size) ** 2
num_heads = 4
embed_dim = 16
mlp_dim = 3072
transformer_units = 8
learning_rate = 0.001

In [63]:
trainset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform= transformation_operation)
valset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform= transformation_operation)


In [64]:
#create train and val batches
train_data = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True)
val_data = torch.utils.data.DataLoader(valset, batch_size=batch_size,
                                          shuffle=False)

Part (i)

In [65]:
class PatchEmbedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embed = nn.Conv2d(num_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x.flatten(2)
        x = x.transpose(1,2)
        return x

In [66]:
class TransformerArchitecture(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm_1 = nn.LayerNorm(embed_dim)
        self.self_attention = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.layer_norm_2 = nn.LayerNorm(embed_dim)
        self.multi_layer_perceptron = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, embed_dim)
        )

    def forward(self, x):
        residual_1 = x
        attention_output = self.self_attention(self.layer_norm_1(x),self.layer_norm_1(x),self.layer_norm_1(x))[0]
        x = attention_output + residual_1
        residual_2 = x
        mlp_output = self.multi_layer_perceptron(self.layer_norm_2(x))
        x = mlp_output + residual_2
        return x

In [67]:
class VisionTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embedding = PatchEmbedding()
        self.cls_token = nn.Parameter(torch.randn(1,1,embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, (img_size // patch_size) ** 2 + 1, embed_dim))
        self.transformer_layers = nn.Sequential(*[TransformerArchitecture() for _ in range(transformer_units)])

        self.mlp_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 10)
        )

    def forward(self,x):
        x = self.patch_embedding(x)
        B = x.size(0)

        cls_tokens = self.cls_token.expand(B , -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        x = self.transformer_layers(x)
        x = x[:,0]
        x = self.mlp_head(x)
        return x

In [68]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VisionTransformer().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [69]:
for epoch in range(5):
    model.train()
    total_loss = 0
    correct_epoch = 0
    total_epoch = 0
    print(f"\nEpoch {epoch+1}")

    for batch_idx, (images, labels) in enumerate(train_data):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss+=loss.item()
        preds = outputs.argmax(dim=1)

        correct = (preds == labels).sum().item()
        accuracy = 100.0 * correct / labels.size(0)

        correct_epoch += correct
        total_epoch += labels.size(0)

        if batch_idx % 100 == 0:
            print(f"  Batch {batch_idx+1:3d}: Loss = {loss.item():.4f}, Accuracy = {accuracy:.2f}%")

    epoch_acc = 100.0 * correct_epoch / total_epoch
    print(f"==> Epoch {epoch+1} Summary: Total Loss = {total_loss:.4f}, Accuracy = {epoch_acc:.2f}%")


Epoch 1
  Batch   1: Loss = 2.3534, Accuracy = 14.06%
  Batch 101: Loss = 2.1590, Accuracy = 14.06%
  Batch 201: Loss = 1.8766, Accuracy = 26.56%
  Batch 301: Loss = 1.5430, Accuracy = 45.31%
  Batch 401: Loss = 1.8500, Accuracy = 21.88%
  Batch 501: Loss = 1.2181, Accuracy = 56.25%
  Batch 601: Loss = 0.8556, Accuracy = 68.75%
  Batch 701: Loss = 1.0088, Accuracy = 71.88%
  Batch 801: Loss = 0.8046, Accuracy = 75.00%
  Batch 901: Loss = 0.4089, Accuracy = 84.38%
==> Epoch 1 Summary: Total Loss = 1259.5081, Accuracy = 50.81%

Epoch 2
  Batch   1: Loss = 0.3650, Accuracy = 89.06%
  Batch 101: Loss = 0.4186, Accuracy = 90.62%
  Batch 201: Loss = 0.2819, Accuracy = 89.06%
  Batch 301: Loss = 0.6780, Accuracy = 82.81%
  Batch 401: Loss = 0.3703, Accuracy = 90.62%
  Batch 501: Loss = 0.4272, Accuracy = 85.94%
  Batch 601: Loss = 0.4396, Accuracy = 89.06%
  Batch 701: Loss = 0.2906, Accuracy = 90.62%
  Batch 801: Loss = 0.1082, Accuracy = 95.31%
  Batch 901: Loss = 0.2977, Accuracy = 89.06%